# Import

In [ ]:
import sys
!{sys.executable} -m pip install lime
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from lime.lime_text import LimeTextExplainer

warnings.filterwarnings("ignore")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=10d77ef31c0bad698e9c4c5bef393b3b0597788388142bba9428d8bea2788c87
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


# Load dataset

In [ ]:
DATA_PATH = ("/content/Labeled Bangla SMS Spam and Ham Dataset for Text Classification and Machine Learning Research (2026).csv")

df = pd.read_csv(DATA_PATH)
df["spamorham"] = df["spamorham"].str.strip().str.lower()
df = df[df["spamorham"].isin(["spam", "ham"])].dropna(subset=["sms"]).reset_index(drop=True)

print(f"✔  Dataset loaded  →  {len(df)} samples")
print(df["spamorham"].value_counts().to_string())
print()

✔  Dataset loaded  →  3998 samples
spamorham
spam    2058
ham     1940



# Text cleaning

In [ ]:
def clean_text(text: str) -> str:
    """Keep only Bangla Unicode characters and whitespace."""
    text = str(text)
    text = re.sub(r"http\S+|www\S+", " ", text)          # remove URLs
    text = re.sub(r"[^\u0980-\u09FF\s]", " ", text)      # keep Bangla letters + spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean"] = df["sms"].apply(clean_text)

# Encode labels  (ham=0, spam=1)

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df["spamorham"])   # ham → 0, spam → 1
print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Label encoding: {'ham': np.int64(0), 'spam': np.int64(1)}


# Train / Test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean"].values, y,
    test_size=0.20, random_state=42, stratify=y
)
print(f"Train : {len(X_train)}  |  Test : {len(X_test)}\n")

Train : 3198  |  Test : 800



# Pipeline: TF-IDF (word unigrams + bigrams) + LinearSVC

In [ ]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        analyzer="word",
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w+\b",   # match any Unicode word
    )),
    ("clf", LinearSVC(max_iter=2000, C=1.0)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

# Evaluation

In [ ]:
acc = accuracy_score(y_test, y_pred)
print("=" * 52)
print(f"  Accuracy : {acc * 100:.2f}%")
print("=" * 52)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel("Predicted", fontsize=13)
ax.set_ylabel("Actual", fontsize=13)
ax.set_title(f"Confusion Matrix  (Accuracy: {acc * 100:.2f}%)", fontsize=14)
plt.tight_layout()

  Accuracy : 93.88%

Classification Report:
              precision    recall  f1-score   support

         ham       0.93      0.95      0.94       388
        spam       0.95      0.93      0.94       412

    accuracy                           0.94       800
   macro avg       0.94      0.94      0.94       800
weighted avg       0.94      0.94      0.94       800



# LIME setup

In [ ]:
def predict_proba(texts):
    """Wrap LinearSVC decision scores into pseudo-probabilities."""
    scores = pipeline.decision_function(texts)
    if scores.ndim == 1:
        scores = np.column_stack([-scores, scores])
    exp = np.exp(scores - scores.max(axis=1, keepdims=True))
    return exp / exp.sum(axis=1, keepdims=True)

SPAM_LABEL = int(le.transform(["spam"])[0])   # should be 1

explainer = LimeTextExplainer(
    class_names=list(le.classes_),
    split_expression=r"\s+",    # split on whitespace → real Bangla words
)

# Visualise LIME for one correct spam and one correct ham example

In [ ]:
def save_lime_chart(text, true_label, pred_label, out_path):
    exp = explainer.explain_instance(
        text, predict_proba,
        num_features=10,
        labels=[SPAM_LABEL],    # always explain toward SPAM class
    )
    features = exp.as_list(label=SPAM_LABEL)

    # Only keep words that push TOWARD spam (positive weight)
    spam_features = [(w, v) for w, v in features if v > 0]
    if not spam_features:
        spam_features = features[:5]   # fallback if all negative

    words, weights = zip(*spam_features)
    fig, ax = plt.subplots(figsize=(9, max(3, len(words) * 0.55)))
    bars = ax.barh(range(len(words)), weights, color="#E53935")
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words, fontsize=13)
    ax.set_xlabel("LIME weight → pushes toward SPAM", fontsize=11)
    ax.set_title(
        f"LIME – Spam indicators\nTrue: {true_label.upper()}  |  Pred: {pred_label.upper()}\n"
        f"SMS: {text[:70]}{'…' if len(text) > 70 else ''}",
        fontsize=12
    )
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


spam_idx = np.where((y_test == SPAM_LABEL) & (y_pred == SPAM_LABEL))[0]
ham_idx  = np.where((y_test == 0)          & (y_pred == 0)          )[0]


# Interactive user input & output

In [ ]:
print("\n" + "═" * 56)
print("  📱  Bangla SMS Spam Detector")
print("═" * 56)
print("Type a Bangla SMS and press Enter to classify it.")
print("Type 'exit' to quit.\n")

while True:
    try:
        sms = input("Enter SMS: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nExiting.")
        break

    if sms.lower() in ("exit", "quit", "q", ""):
        print("Goodbye!")
        break

    cleaned = clean_text(sms)
    if not cleaned:
        print("⚠  Could not extract Bangla text. Please type in Bangla.\n")
        continue

    pred_idx  = pipeline.predict([cleaned])[0]
    pred_label = le.classes_[pred_idx]
    proba      = predict_proba([cleaned])[0]
    confidence = proba[pred_idx] * 100

    print(f"\n  Result     : {'🚨 SPAM' if pred_label == 'spam' else '✅ HAM (not spam)'}")
    print(f"  Confidence : {confidence:.1f}%")

    # Show LIME only for spam predictions (or always, showing spam-pushing words)
    exp = explainer.explain_instance(
        cleaned, predict_proba,
        num_features=10,
        labels=[SPAM_LABEL],
    )
    spam_words = [(w, v) for w, v in exp.as_list(label=SPAM_LABEL) if v > 0]

    if spam_words:
        print("\n  Words pushing toward SPAM:")
        max_w = max(v for _, v in spam_words)
        for word, weight in sorted(spam_words, key=lambda x: -x[1]):
            bar_len = int((weight / max_w) * 20)
            bar = "█" * bar_len
            print(f"    {word:<20s}  {bar}  ({weight:.4f})")
    else:
        print("  (No strong spam-indicator words found in this message.)")

    print()


════════════════════════════════════════════════════════
  📱  Bangla SMS Spam Detector
════════════════════════════════════════════════════════
Type a Bangla SMS and press Enter to classify it.
Type 'exit' to quit.

Enter SMS: আপনি ১০ লক্ষ টাকা জিতেছেন!

  Result     : 🚨 SPAM
  Confidence : 53.4%

  Words pushing toward SPAM:
    আপনি                  ████████████████████  (0.1174)
    ১০                    ██████████████  (0.0864)
    টাকা                  █████████  (0.0538)
    জিতেছেন               █████  (0.0319)

Enter SMS: তুমি কোথায়?

  Result     : ✅ HAM (not spam)
  Confidence : 82.4%

  Words pushing toward SPAM:
    কোথায়                 ████████████████████  (0.1273)

Enter SMS: ফ্রি রিচার্জ পেতে এখনই কল করুন!

  Result     : 🚨 SPAM
  Confidence : 98.2%

  Words pushing toward SPAM:
    করুন                  ████████████████████  (0.1010)
    রিচার্জ               █████████████████  (0.0882)
    ফ্রি                  ███████████  (0.0564)
    কল                    ███  (0